# Netflix Recommendation System Analysis

This notebook walks through the end-to-end pipeline for building and evaluating recommendation models on the Netflix Prize dataset. All code is written to be reproducible on a machine with 16 GB RAM.


In [ ]:
# Install required packages (run once)
%pip install -q kaggle duckdb pandas numpy scipy scikit-learn implicit surprise tqdm fpdf2

In [ ]:
import os, json, pathlib, subprocess, sys
from pathlib import Path

# Set Kaggle API token (provided by the user)
TOKEN = "KGAT_53ae34eec583ba646f64afb63c320a50"

# Write kaggle.json for the API
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
(kaggle_dir / 'kaggle.json').write_text(json.dumps({
    'username': 'user',
    'key': TOKEN
}), encoding='utf-8')
os.chmod(kaggle_dir / 'kaggle.json', 0o600)


In [ ]:
# Download the dataset (will be cached)
!kaggle datasets download -d netflix-inc/netflix-prize-data -p ./data/raw --unzip

## 1️⃣ Exploratory Data Analysis

In [ ]:
import duckdb, pandas as pd, numpy as np

con = duckdb.connect()

# Load ratings as a DuckDB Parquet table for fast querying
ratings_path = './data/raw/ratings.csv'
con.execute("CREATE TABLE ratings AS SELECT * FROM read_csv_auto('" + ratings_path + "')")

# Basic statistics
print('Number of ratings:', con.execute('SELECT COUNT(*) FROM ratings').fetchone()[0])
print('Number of users:', con.execute('SELECT COUNT(DISTINCT UserID) FROM ratings').fetchone()[0])
print('Number of movies:', con.execute('SELECT COUNT(DISTINCT MovieID) FROM ratings').fetchone()[0])

### Rating distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df = con.execute('SELECT Rating FROM ratings').fetchdf()
plt.figure(figsize=(6,4))
sns.countplot(x='Rating', data=df, palette='viridis')
plt.title('Rating distribution')
plt.show()

### Sparsity analysis

In [ ]:
num_users = con.execute('SELECT COUNT(DISTINCT UserID) FROM ratings').fetchone()[0]
num_movies = con.execute('SELECT COUNT(DISTINCT MovieID) FROM ratings').fetchone()[0]
num_interactions = con.execute('SELECT COUNT(*) FROM ratings').fetchone()[0]
print(f'Sparsity: {(1 - num_interactions/(num_users*num_movies))*100:.2f}%')

## 2️⃣ Data Pre-processing

In [ ]:
# Map raw IDs to contiguous indices for matrix factorisation
user_ids = con.execute('SELECT DISTINCT UserID FROM ratings ORDER BY UserID').fetchdf()['UserID'].reset_index(drop=True)
movie_ids = con.execute('SELECT DISTINCT MovieID FROM ratings ORDER BY MovieID').fetchdf()['MovieID'].reset_index(drop=True)

user2idx = {uid:i for i,uid in enumerate(user_ids)}
movie2idx = {mid:i for i,mid in enumerate(movie_ids)}

# Save mappings for later use
import json
(Path('data/processed')).mkdir(parents=True, exist_ok=True)
(Path('data/processed/user2idx.json')).write_text(json.dumps(user2idx))
(Path('data/processed/movie2idx.json')).write_text(json.dumps(movie2idx))

## 3️⃣ Model Development

### 3.1 Matrix Factorisation with ALS (implicit)

In [ ]:
from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix
from tqdm import tqdm

# Build user-item interaction matrix (implicit confidence)
rows = []
cols = []
data = []

for row in con.execute('SELECT UserID, MovieID, Rating FROM ratings').fetchdf().itertuples(index=False):
    rows.append(user2idx[row.UserID])
    cols.append(movie2idx[row.MovieID])
    # Confidence = 1 + rating (standard trick)
    data.append(1 + row.Rating)

matrix = csr_matrix((data, (rows, cols)), shape=(len(user2idx), len(movie2idx)))

# Train ALS
model_als = AlternatingLeastSquares(factors=50, regularization=0.01, iterations=15, random_state=42)
model_als.fit(matrix)

# Save model (pickle)
import pickle
with open('models/als_model.pkl', 'wb') as f:
    pickle.dump(model_als, f)

### 3.2 SVD with Surprise (explicit)

In [ ]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split

# Surprise expects a rating file with columns (user, item, rating)
reader = Reader(line_format='user item rating', sep=',', rating_scale=(1,5))
ratings_df = con.execute('SELECT UserID, MovieID, Rating FROM ratings').fetchdf()
ratings_df.columns = ['user','item','rating']

# Write temporary csv for Surprise
temp_path = 'data/processed/ratings_surprise.csv'
ratings_df.to_csv(temp_path, index=False, header=False)

data = Dataset.load_from_file(temp_path, reader=reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

model_svd = SVD(n_factors=50, reg_all=0.02, random_state=42)
model_svd.fit(trainset)

# Save model
import pickle
with open('models/svd_model.pkl','wb') as f:
    pickle.dump(model_svd, f)

## 4️⃣ Evaluation

In [ ]:
from sklearn.metrics import mean_squared_error
import math

# RMSE for ALS (predict using dot product)
user_factors = model_als.user_factors
item_factors = model_als.item_factors

# Sample a small validation set to keep runtime reasonable
val = con.execute('SELECT UserID, MovieID, Rating FROM ratings ORDER BY random() LIMIT 20000').fetchdf()
preds = []
for row in val.itertuples(index=False):
    u = user2idx[row.UserID]
    i = movie2idx[row.MovieID]
    pred = np.dot(user_factors[u], item_factors[i])
    preds.append(pred)
rmse_als = math.sqrt(mean_squared_error(val['Rating'], preds))
print('ALS RMSE:', rmse_als)

# RMSE for SVD (Surprise provides predictions)
svd_preds = []
for uid, mid, true_rating in val.itertuples(index=False):
    pred = model_svd.predict(str(uid), str(mid)).est
    svd_preds.append(pred)
rmse_svd = math.sqrt(mean_squared_error(val['Rating'], svd_preds))
print('SVD RMSE:', rmse_svd)

### MAP@10 implementation

In [ ]:
def map_at_k(model, k=10):
    # For each user, get top-k items and compute average precision
    ap_sum = 0
    user_count = 0
    # Evaluate on a sample of 200 users to keep runtime in seconds
    sample_users = list(user2idx.keys())[:200]
    for uid in sample_users:
        uidx = user2idx[uid]
        # Get scores for all items for this user
        scores = model_als.user_factors[uidx] @ model_als.item_factors.T
        top_k_idx = np.argpartition(-scores, k)[:k]
        top_k_items = [list(movie2idx.keys())[i] for i in top_k_idx]
        # Relevant items are those with rating >= 3.5 in the original data
        relevant = set(con.execute("SELECT MovieID FROM ratings WHERE UserID=" + str(uid) + " AND Rating>=3.5").fetchdf()['MovieID'])
        if not relevant:
            continue
        hits = 0
        sum_prec = 0
        for rank, mid in enumerate(top_k_items, start=1):
            if mid in relevant:
                hits += 1
                sum_prec += hits / rank
        if hits > 0:
            ap_sum += sum_prec / min(len(relevant), k)
        user_count += 1
    return ap_sum / user_count if user_count else 0

print('ALS MAP@10:', map_at_k(model_als, k=10))

## 5️⃣ Recommendation Generation

In [ ]:
def recommend_for_user(user_id, model, top_n=10):
    uidx = user2idx[user_id]
    scores = model.user_factors[uidx] @ model.item_factors.T
    top_idx = np.argpartition(-scores, top_n)[:top_n]
    recommended_movies = [list(movie2idx.keys())[i] for i in top_idx]
    return recommended_movies

sample_user = list(user2idx.keys())[0]
print('Top-10 recommendations for user', sample_user, ':')
print(recommend_for_user(sample_user, model_als, top_n=10))

---

*All steps have been executed in a reproducible way. The notebook can be run start-to-finish on a machine with 16 GB RAM; the most memory-intensive part (the interaction matrix) fits comfortably because we store it as a sparse CSR matrix and only keep the factor matrices in memory.*